[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Depth-Anything-3_Colab.ipynb)  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Depth-Anything-3_Colab.ipynb)

# 🌊 ByteDance Depth Anything 3 (DA3) — Any-View Metric Depth, Camera Pose, `.glb` & 3DGS `.ply`

**[Depth Anything 3 (DA3)](https://depth-anything-3.github.io)** ([arXiv:2511.10647](https://arxiv.org/abs/2511.10647)) by the ByteDance Seed Team recovers spatially consistent 3D visual geometry from **any number of views** (a single image, unposed multi-view photo sets, or video clips) in a single feed-forward pass. Using a plain DINOv2 ViT backbone and a unified **depth-ray representation**, DA3 jointly predicts **metric/relative depth maps**, **camera extrinsics & intrinsics**, **3D point clouds with camera wireframes (`.glb`)**, and **feed-forward 3D Gaussian Splatting scenes (`.ply` + `.mp4`)**.

* **Paper:** [Depth Anything 3: Recovering the Visual Space from Any Views (arXiv:2511.10647)](https://arxiv.org/abs/2511.10647)
* **Code:** [github.com/ByteDance-Seed/Depth-Anything-3](https://github.com/ByteDance-Seed/Depth-Anything-3) &nbsp;·&nbsp; **Project:** [depth-anything-3.github.io](https://depth-anything-3.github.io) &nbsp;·&nbsp; **Space:** [🤗 depth-anything/Depth-Anything-3](https://huggingface.co/spaces/depth-anything/Depth-Anything-3)
* **Refreshed `-1.1` Checkpoints Included:** Uses the updated `-1.1` models (`DA3NESTED-GIANT-LARGE-1.1`, `DA3-GIANT-1.1`, `DA3-LARGE-1.1`) retrained after fixing the upstream street-scene bug.

---

### 🗂️ Supported Models & Licenses

| Model ID | Params | Metric Depth | Multi-View Pose | Native 3DGS Head | Sky Seg | License |
|---|---|---|---|---|---|---|
| **`depth-anything/DA3NESTED-GIANT-LARGE-1.1`** *(Default Flagship)* | 1.40B | ✅ Real-world meters | ✅ Extrinsics + Intrinsics | ✅ Direct 3DGS `.ply` | ✅ | CC BY-NC 4.0 |
| **`depth-anything/DA3METRIC-LARGE`** *(Commercial-OK Metric)* | 0.35B | ✅ Real-world meters | Pinhole unprojection | Point-to-3DGS `.ply` | ✅ | **Apache 2.0 (Commercial-OK)** |
| **`depth-anything/DA3-LARGE-1.1`** *(Multi-View Depth + Pose)* | 0.35B | Relative scale | ✅ Extrinsics + Intrinsics | Point-to-3DGS `.ply` | — | CC BY-NC 4.0 |
| **`depth-anything/DA3-BASE`** *(Commercial-OK Multi-View + Pose)* | 0.12B | Relative scale | ✅ Extrinsics + Intrinsics | Point-to-3DGS `.ply` | — | **Apache 2.0 (Commercial-OK)** |
| **`depth-anything/DA3-GIANT-1.1`** *(Any-View + 3DGS)* | 1.15B | Relative scale | ✅ Extrinsics + Intrinsics | ✅ Direct 3DGS `.ply` | — | CC BY-NC 4.0 |

---

### 🖥️ GPU & Memory Guide

| GPU | VRAM | Supported Models | Typical Multi-View / Video Capacity |
|---|---|---|---|
| **T4** | 15 GB | All models (`DA3METRIC-LARGE`, `DA3-LARGE-1.1`, `DA3NESTED-GIANT-LARGE-1.1`) | 1–24 views at `504px` in ~3–15 s |
| **L4** | 24 GB | All models + high-res 3DGS video rendering | 1–48 views at `504px` in ~2–10 s |
| **A100** | 40 GB | All models + large multi-view batches | 1–96+ views in ~1–8 s |

In [ ]:
# @title STEP 1 — Install Depth Anything 3 & Dependencies
# @markdown Clones [ByteDance-Seed/Depth-Anything-3](https://github.com/ByteDance-Seed/Depth-Anything-3), installs the `depth_anything_3` package (`da3` CLI + Python API), `gsplat` for 3D Gaussian rendering, and `trimesh`/`plyfile` for `.glb` and `.ply` export.

import os, sys, subprocess, shutil, pathlib
import torch
from tqdm.auto import tqdm

print("=" * 72)
print("Depth Anything 3 (DA3) — STEP 1: Install Dependencies")
print("=" * 72)
print(f"  Python : {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}  (CUDA: {torch.version.cuda})")

if not torch.cuda.is_available():
    raise SystemExit("[ERROR] No GPU detected. Please switch your Colab runtime to a GPU (T4, L4, or A100).")

CONNECT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
if CONNECT_GOOGLE_DRIVE and not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as _e:
        print(f"[Drive] Mount skipped: {_e}")

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/DepthAnything3') if os.path.exists('/content/drive/MyDrive') else pathlib.Path('/content/AEI_3D_Cache/DepthAnything3')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.makedirs(os.environ['HUGGINGFACE_HUB_CACHE'], exist_ok=True)
os.makedirs('/content/da3_inputs', exist_ok=True)
print(f"[Cache] HF_HOME -> {os.environ['HF_HOME']}")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"[GPU] {gpu_name} ({vram_gb:.1f} GB VRAM)")

REPO_DIR = '/content/Depth-Anything-3'
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print("[Git] Cloning https://github.com/ByteDance-Seed/Depth-Anything-3.git ...")
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ByteDance-Seed/Depth-Anything-3.git', REPO_DIR], check=True)
else:
    print("[Git] Updating /content/Depth-Anything-3 (git pull --ff-only)...")
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

# Install core Python packages and Gradio with version pin
pkgs = [
    "gradio==5.49.1",
    "tqdm==4.66.5",
    "huggingface_hub",
    "safetensors",
    "omegaconf",
    "addict",
    "einops",
    "e3nn",
    "evo",
    "moviepy==1.0.3",
    "plyfile",
    "trimesh",
    "pygltflib",
    "pillow_heif",
    "opencv-python-headless",
    "imageio",
    "imageio-ffmpeg",
    "scipy",
    "matplotlib",
    "typer>=0.9.0",
    "uvicorn",
    "fastapi",
]
print("[Pip] Installing DA3 dependencies...")
for chunk_start in tqdm(range(0, len(pkgs), 8), desc="Installing packages"):
    chunk = pkgs[chunk_start:chunk_start + 8]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + chunk, check=True)

# Optional pycolmap + gsplat for COLMAP and 3DGS novel-view video rendering
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycolmap", "gsplat"], check=False)
except Exception as _e:
    print(f"[Optional] gsplat/pycolmap install note: {_e}")

# Install Depth-Anything-3 in editable mode (--no-deps so it doesn't downgrade torch/gradio)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR], check=True)

src_dir = os.path.join(REPO_DIR, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from depth_anything_3.api import DepthAnything3
from depth_anything_3.registry import MODEL_REGISTRY
print(f"[Verify] depth_anything_3 imported successfully! Registered presets: {list(MODEL_REGISTRY.keys())}")
print(f"[DONE] STEP 1 complete on {gpu_name} ({vram_gb:.1f} GB VRAM) — proceed to STEP 2.")

In [ ]:
# @title STEP 2 — Pre-Download & Cache Depth Anything 3 Checkpoints on Google Drive
# @markdown Downloads your selected Depth Anything 3 model checkpoints (`DA3NESTED-GIANT-LARGE-1.1`, `DA3METRIC-LARGE`, `DA3-LARGE-1.1`, or `DA3-BASE`) to `/content/drive/MyDrive/AEI_3D_Cache/DepthAnything3`.

import os, sys, pathlib
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/DepthAnything3') if os.path.exists('/content/drive/MyDrive') else pathlib.Path('/content/AEI_3D_Cache/DepthAnything3')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT / 'huggingface' / 'hub')
os.makedirs(os.environ['HUGGINGFACE_HUB_CACHE'], exist_ok=True)

DEFAULT_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"  # @param ["depth-anything/DA3NESTED-GIANT-LARGE-1.1", "depth-anything/DA3METRIC-LARGE", "depth-anything/DA3-LARGE-1.1", "depth-anything/DA3-BASE", "depth-anything/DA3-GIANT-1.1"]
ALSO_CACHE_METRIC_LARGE = True  # @param {type:"boolean"}

models_to_cache = [DEFAULT_MODEL_ID]
if ALSO_CACHE_METRIC_LARGE and "depth-anything/DA3METRIC-LARGE" not in models_to_cache:
    models_to_cache.append("depth-anything/DA3METRIC-LARGE")

print("=" * 72)
print("STEP 2 — Caching Depth Anything 3 Checkpoints to Google Drive")
print("=" * 72)
print(f"  HF_HOME : {os.environ['HF_HOME']}")

for repo_id in models_to_cache:
    print(f"\n[Cache] Downloading {repo_id} ...")
    try:
        local_path = snapshot_download(
            repo_id=repo_id,
            allow_patterns=["*.json", "*.safetensors", "*.yaml", "*.md"],
            tqdm_class=tqdm,
            max_workers=4,
        )
        print(f"  ✓ Cached {repo_id} -> {local_path}")
    except Exception as e:
        print(f"  [WARN] Could not pre-cache {repo_id}: {e}")

ex_root = pathlib.Path('/content/Depth-Anything-3/assets/examples')
if ex_root.exists():
    print(f"\n[Assets] Found built-in DA3 examples in {ex_root}: {[p.name for p in ex_root.iterdir()]}")
os.makedirs('/content/da3_outputs', exist_ok=True)
print("[DONE] STEP 2 complete — proceed to STEP 3.")

In [ ]:
# @title STEP 3 — Core Depth Anything 3 Engine (Single-View, Multi-View & Video $\rightarrow$ `.glb`, 3DGS `.ply`, Depth & Poses)
# @markdown Defines `get_da3_model()`, `extract_video_frames()`, `export_pointcloud_as_3dgs_ply()`, and `run_da3_inference()` supporting all DA3 model families.

import os, sys, gc, time, json, glob, math, shutil, pathlib, traceback
from datetime import datetime
import numpy as np
import torch
import cv2
import imageio
import trimesh
from PIL import Image
from plyfile import PlyData, PlyElement

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_3D_Cache/DepthAnything3') if os.path.exists('/content/drive/MyDrive') else pathlib.Path('/content/AEI_3D_Cache/DepthAnything3')
os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(CACHE_ROOT / 'huggingface' / 'hub'))

REPO_DIR = '/content/Depth-Anything-3'
src_dir = os.path.join(REPO_DIR, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from depth_anything_3.api import DepthAnything3
from depth_anything_3.utils.export.glb import export_to_glb
from depth_anything_3.utils.export.depth_vis import export_to_depth_vis
from depth_anything_3.utils.export.npz import export_to_npz, export_to_mini_npz
from depth_anything_3.utils.export.gs import export_to_gs_ply, export_to_gs_video
from depth_anything_3.utils.visualize import visualize_depth

DA3_MODEL_SPECS = {
    "depth-anything/DA3NESTED-GIANT-LARGE-1.1": {
        "label": "DA3NESTED-GIANT-LARGE-1.1 (1.40B | Any-View Metric Depth + Pose + 3DGS .ply | CC BY-NC 4.0)",
        "supports_gs": True,
        "supports_pose": True,
        "is_metric": True,
        "license": "CC BY-NC 4.0",
    },
    "depth-anything/DA3METRIC-LARGE": {
        "label": "DA3METRIC-LARGE (0.35B | Monocular Metric Depth | Apache 2.0 Commercial-OK)",
        "supports_gs": False,
        "supports_pose": False,
        "is_metric": True,
        "license": "Apache 2.0",
    },
    "depth-anything/DA3-LARGE-1.1": {
        "label": "DA3-LARGE-1.1 (0.35B | Multi-View Depth + Camera Pose | CC BY-NC 4.0)",
        "supports_gs": False,
        "supports_pose": True,
        "is_metric": False,
        "license": "CC BY-NC 4.0",
    },
    "depth-anything/DA3-BASE": {
        "label": "DA3-BASE (0.12B | Fast Multi-View Depth + Pose | Apache 2.0 Commercial-OK)",
        "supports_gs": False,
        "supports_pose": True,
        "is_metric": False,
        "license": "Apache 2.0",
    },
    "depth-anything/DA3-GIANT-1.1": {
        "label": "DA3-GIANT-1.1 (1.15B | Any-View Depth + Pose + 3DGS .ply | CC BY-NC 4.0)",
        "supports_gs": True,
        "supports_pose": True,
        "is_metric": False,
        "license": "CC BY-NC 4.0",
    },
}

_LOADED_MODEL_ID = None
_LOADED_MODEL = None

def cleanup_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_da3_model(model_id="depth-anything/DA3NESTED-GIANT-LARGE-1.1"):
    """Lazy-load and cache the requested DepthAnything3 model on GPU."""
    global _LOADED_MODEL_ID, _LOADED_MODEL
    if _LOADED_MODEL is not None and _LOADED_MODEL_ID == model_id:
        return _LOADED_MODEL
    if _LOADED_MODEL is not None:
        print(f"[DA3] Unloading previous model ({_LOADED_MODEL_ID})...")
        del _LOADED_MODEL
        _LOADED_MODEL = None
        cleanup_vram()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[DA3] Loading {model_id} onto {device} ...")
    t0 = time.time()
    model = DepthAnything3.from_pretrained(model_id)
    model = model.to(device=device)
    model.eval()
    _LOADED_MODEL = model
    _LOADED_MODEL_ID = model_id
    print(f"[DA3] Loaded {model_id} in {time.time() - t0:.1f}s")
    return model

def extract_video_frames(video_path, out_dir, sample_fps=2.0, max_frames=24):
    """Extract evenly spaced RGB frames from a video file using OpenCV."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(1, int(round(src_fps / max(0.25, float(sample_fps)))))

    saved_paths = []
    frame_idx = 0
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if frame_idx % step == 0:
            out_p = os.path.join(out_dir, f"frame_{len(saved_paths):04d}.png")
            cv2.imwrite(out_p, frame_bgr)
            saved_paths.append(out_p)
            if len(saved_paths) >= int(max_frames):
                break
        frame_idx += 1
    cap.release()
    print(f"[Video] Extracted {len(saved_paths)} frame(s) from {os.path.basename(str(video_path))} (src_fps={src_fps:.1f}, total={total_frames})")
    return saved_paths

def _ensure_camera_and_conf_for_export(prediction):
    """Populate fallback pinhole intrinsics, w2c extrinsics, and confidence for monocular-only models (e.g., DA3METRIC-LARGE)."""
    N, H, W = prediction.depth.shape
    if prediction.conf is None:
        prediction.conf = np.ones((N, H, W), dtype=np.float32) * 2.0
    if prediction.intrinsics is None:
        f = float(max(H, W))
        K = np.array([[f, 0.0, W / 2.0], [0.0, f, H / 2.0], [0.0, 0.0, 1.0]], dtype=np.float32)
        prediction.intrinsics = np.repeat(K[None, :, :], N, axis=0)
    if prediction.extrinsics is None:
        ext = np.eye(4, dtype=np.float32)[:3, :]
        exts = np.repeat(ext[None, :, :], N, axis=0)
        for i in range(N):
            exts[i, 0, 3] = float(i) * 0.25
        prediction.extrinsics = exts
    return prediction

def export_pointcloud_as_3dgs_ply(prediction, save_path, max_gaussians=250000):
    """Convert unprojected DA3 depth + RGB maps into a standard 3D Gaussian Splatting .ply file when native infer_gs is unavailable."""
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    prediction = _ensure_camera_and_conf_for_export(prediction)
    N, H, W = prediction.depth.shape
    imgs = prediction.processed_images  # (N, H, W, 3) uint8

    all_pts = []
    all_rgb = []
    all_scales = []

    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    for i in range(N):
        d = prediction.depth[i]
        K = prediction.intrinsics[i]
        ext = prediction.extrinsics[i]
        w2c = np.eye(4, dtype=np.float32)
        w2c[:3, :4] = ext[:3, :4]
        c2w = np.linalg.inv(w2c)

        valid = np.isfinite(d) & (d > 1e-3)
        if not np.any(valid):
            continue
        z = d[valid]
        x = (u_grid[valid] - K[0, 2]) * z / max(K[0, 0], 1e-5)
        y = (v_grid[valid] - K[1, 2]) * z / max(K[1, 1], 1e-5)
        pts_cam = np.stack([x, y, z, np.ones_like(z)], axis=-1)  # (M, 4)
        pts_world = (pts_cam @ c2w.T)[:, :3]
        # Convert OpenCV camera convention to standard 3DGS viewer orientation
        pts_world[:, 1] *= -1.0
        pts_world[:, 2] *= -1.0

        rgb = imgs[i][valid].astype(np.float32) / 255.0
        px_scale = np.clip(z / max(K[0, 0], 1e-5) * 1.2, 1e-4, 0.1)
        log_scale = np.log(px_scale)[:, None].repeat(3, axis=1)

        all_pts.append(pts_world)
        all_rgb.append(rgb)
        all_scales.append(log_scale)

    if not all_pts:
        raise RuntimeError("No valid depth points available to construct 3DGS PLY.")

    xyz = np.concatenate(all_pts, axis=0).astype(np.float32)
    rgb = np.concatenate(all_rgb, axis=0).astype(np.float32)
    scales = np.concatenate(all_scales, axis=0).astype(np.float32)

    if xyz.shape[0] > max_gaussians:
        rng = np.random.default_rng(42)
        idx = rng.choice(xyz.shape[0], size=max_gaussians, replace=False)
        xyz, rgb, scales = xyz[idx], rgb[idx], scales[idx]

    C0 = 0.28209479177387814
    f_dc = ((rgb - 0.5) / C0).astype(np.float32)
    M = xyz.shape[0]
    normals = np.zeros((M, 3), dtype=np.float32)
    opacity = np.full((M, 1), 2.2, dtype=np.float32)  # inverse_sigmoid(~0.9)
    rot = np.zeros((M, 4), dtype=np.float32)
    rot[:, 0] = 1.0

    dtype_full = [
        ("x", "f4"), ("y", "f4"), ("z", "f4"),
        ("nx", "f4"), ("ny", "f4"), ("nz", "f4"),
        ("f_dc_0", "f4"), ("f_dc_1", "f4"), ("f_dc_2", "f4"),
        ("opacity", "f4"),
        ("scale_0", "f4"), ("scale_1", "f4"), ("scale_2", "f4"),
        ("rot_0", "f4"), ("rot_1", "f4"), ("rot_2", "f4"), ("rot_3", "f4"),
    ]
    elements = np.empty(M, dtype=dtype_full)
    attrs = np.concatenate([xyz, normals, f_dc, opacity, scales, rot], axis=1)
    for idx_col, (name, _) in enumerate(dtype_full):
        elements[name] = attrs[:, idx_col]
    PlyData([PlyElement.describe(elements, "vertex")]).write(save_path)
    return save_path

def resolve_da3_inputs(input_source, work_dir, video_fps=2.0, max_frames=24):
    """Resolve a single image path, list of image paths, directory of images, or video into a list of image paths."""
    VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}
    IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".heic"}

    if isinstance(input_source, (list, tuple)):
        paths = []
        for item in input_source:
            p = item.name if hasattr(item, "name") else str(item)
            if os.path.splitext(p)[1].lower() in VIDEO_EXTS:
                return extract_video_frames(p, os.path.join(work_dir, "video_frames"), sample_fps=video_fps, max_frames=max_frames)
            if os.path.isfile(p):
                paths.append(p)
        return paths

    if isinstance(input_source, Image.Image):
        tmp_p = os.path.join(work_dir, "input_0000.png")
        os.makedirs(work_dir, exist_ok=True)
        input_source.save(tmp_p)
        return [tmp_p]

    p_str = str(input_source).strip()
    if os.path.isdir(p_str):
        files = sorted(
            str(f) for f in pathlib.Path(p_str).iterdir()
            if f.is_file() and f.suffix.lower() in IMG_EXTS
        )
        return files[:max_frames] if max_frames else files

    if os.path.isfile(p_str):
        ext = os.path.splitext(p_str)[1].lower()
        if ext in VIDEO_EXTS:
            return extract_video_frames(p_str, os.path.join(work_dir, "video_frames"), sample_fps=video_fps, max_frames=max_frames)
        return [p_str]

    raise FileNotFoundError(f"Could not find input image, folder, or video at: {input_source}")

def run_da3_inference(
    input_source,
    model_id="depth-anything/DA3NESTED-GIANT-LARGE-1.1",
    process_res=504,
    process_res_method="upper_bound_resize",
    ref_view_strategy="saddle_balanced",
    use_ray_pose=False,
    conf_thresh_percentile=40.0,
    num_max_points=1000000,
    show_cameras=True,
    export_gs_ply_flag=True,
    export_gs_video_flag=False,
    video_fps=2.0,
    max_frames=24,
    output_dir=None,
):
    """Run Depth Anything 3 inference and export .glb, 3DGS .ply, colormap depth maps, and camera poses (.npz + .json)."""
    t0 = time.time()
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    if output_dir is None:
        output_dir = os.path.join("/content/da3_outputs", f"da3_{ts}")
    os.makedirs(output_dir, exist_ok=True)

    image_paths = resolve_da3_inputs(input_source, output_dir, video_fps=video_fps, max_frames=max_frames)
    if not image_paths:
        raise ValueError("No valid input images found for Depth Anything 3.")

    spec = DA3_MODEL_SPECS.get(model_id, {"supports_gs": "GIANT" in model_id.upper()})
    can_infer_gs = bool(spec.get("supports_gs", False)) and (export_gs_ply_flag or export_gs_video_flag)

    model = get_da3_model(model_id)
    print(f"[DA3] Running inference on {len(image_paths)} view(s) | model={model_id} | res={process_res} | infer_gs={can_infer_gs}")
    prediction = model.inference(
        image=image_paths,
        infer_gs=can_infer_gs,
        use_ray_pose=bool(use_ray_pose),
        ref_view_strategy=str(ref_view_strategy),
        process_res=int(process_res),
        process_res_method=str(process_res_method),
        export_dir=None,
    )

    # Ensure monocular models (like DA3METRIC-LARGE) have camera matrices for GLB & pose export
    prediction = _ensure_camera_and_conf_for_export(prediction)

    # 1. Export Colormap Depth Maps
    export_to_depth_vis(prediction, output_dir)
    depth_vis_files = sorted(glob.glob(os.path.join(output_dir, "depth_vis", "*.jpg")))

    # 2. Export 3D Point Cloud + Camera Frustums GLB
    glb_path = export_to_glb(
        prediction,
        export_dir=output_dir,
        num_max_points=int(num_max_points),
        conf_thresh_percentile=float(conf_thresh_percentile),
        show_cameras=bool(show_cameras),
        export_depth_vis=False,
    )

    # 3. Export 3D Gaussian Splatting (.ply) & optional 3DGS trajectory video (.mp4)
    ply_path = None
    gs_video_path = None
    if export_gs_ply_flag:
        if can_infer_gs and prediction.gaussians is not None:
            try:
                export_to_gs_ply(prediction, output_dir)
                cand = sorted(glob.glob(os.path.join(output_dir, "gs_ply", "*.ply")))
                if cand:
                    ply_path = cand[0]
            except Exception as e:
                print(f"[DA3] Native export_to_gs_ply fallback ({e}); building point-based 3DGS PLY...")
        if ply_path is None:
            os.makedirs(os.path.join(output_dir, "gs_ply"), exist_ok=True)
            ply_path = export_pointcloud_as_3dgs_ply(
                prediction,
                os.path.join(output_dir, "gs_ply", "0000.ply"),
                max_gaussians=min(int(num_max_points), 300000),
            )

    if export_gs_video_flag and can_infer_gs and prediction.gaussians is not None:
        try:
            export_to_gs_video(prediction, output_dir)
            vcand = sorted(glob.glob(os.path.join(output_dir, "gs_video", "*.mp4")))
            if vcand:
                gs_video_path = vcand[0]
        except Exception as e:
            print(f"[DA3] 3DGS video render skipped (gsplat required): {e}")

    # 4. Export NPZ + Camera Poses JSON
    export_to_mini_npz(prediction, output_dir)
    npz_path = os.path.join(output_dir, "exports", "mini_npz", "results.npz")
    if not os.path.isfile(npz_path):
        cand_npz = sorted(glob.glob(os.path.join(output_dir, "**", "*.npz"), recursive=True))
        npz_path = cand_npz[0] if cand_npz else None

    poses_json_path = os.path.join(output_dir, "camera_poses.json")
    poses_payload = {
        "model_id": model_id,
        "license": spec.get("license", "Unknown"),
        "num_views": int(prediction.depth.shape[0]),
        "resolution_hw": [int(prediction.depth.shape[1]), int(prediction.depth.shape[2])],
        "is_metric": bool(prediction.is_metric or spec.get("is_metric", False)),
        "scale_factor": float(prediction.scale_factor) if prediction.scale_factor is not None else 1.0,
        "depth_min": float(np.nanmin(prediction.depth)),
        "depth_max": float(np.nanmax(prediction.depth)),
        "intrinsics_K": prediction.intrinsics.tolist(),
        "extrinsics_w2c": prediction.extrinsics.tolist(),
    }
    with open(poses_json_path, "w", encoding="utf-8") as f:
        json.dump(poses_payload, f, indent=2)

    cleanup_vram()
    elapsed = time.time() - t0
    print(f"[DA3] Done in {elapsed:.1f}s -> GLB: {glb_path} | 3DGS PLY: {ply_path}")
    return {
        "output_dir": output_dir,
        "glb_path": glb_path,
        "ply_path": ply_path,
        "gs_video_path": gs_video_path,
        "npz_path": npz_path,
        "poses_json_path": poses_json_path,
        "depth_vis_files": depth_vis_files,
        "poses_summary": poses_payload,
        "elapsed_s": round(elapsed, 2),
    }

print("[Step 3] Depth Anything 3 core engine ready.")

In [ ]:
# @title STEP 4 — Launch Gradio UI (Any-View Depth, Pose, `.glb` & 3DGS `.ply`)
# @markdown Interactive Gradio interface supporting **Single-Image**, **Multi-View Image Upload**, and **Video Frame Extraction** across `DA3NESTED-GIANT-LARGE-1.1`, `DA3METRIC-LARGE` (Apache 2.0), `DA3-LARGE-1.1`, and `DA3-BASE` (Apache 2.0).

import os, sys, glob, traceback
import gradio as gr
from IPython.display import clear_output

MODEL_CHOICES = list(DA3_MODEL_SPECS.keys())
DEFAULT_MODEL = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"

ex_soh_dir = "/content/Depth-Anything-3/assets/examples/SOH"
EXAMPLE_SCENES = []
if os.path.isdir(ex_soh_dir):
    soh_imgs = sorted(glob.glob(os.path.join(ex_soh_dir, "*.*")))
    if soh_imgs:
        EXAMPLE_SCENES.append([soh_imgs[0], DEFAULT_MODEL])

def _ui_run_images(
    single_img,
    multi_files,
    model_id,
    process_res,
    ref_view_strategy,
    use_ray_pose,
    conf_thresh_percentile,
    num_max_points,
    show_cameras,
    export_gs_ply_flag,
    export_gs_video_flag,
    progress=gr.Progress(track_tqdm=True),
):
    if multi_files:
        src = multi_files
    elif single_img is not None:
        src = single_img
    else:
        raise gr.Error("Please upload a single image or 2+ multi-view images.")
    try:
        progress(0.1, desc=f"Running {model_id}...")
        res = run_da3_inference(
            input_source=src,
            model_id=model_id,
            process_res=int(process_res),
            ref_view_strategy=ref_view_strategy,
            use_ray_pose=bool(use_ray_pose),
            conf_thresh_percentile=float(conf_thresh_percentile),
            num_max_points=int(num_max_points),
            show_cameras=bool(show_cameras),
            export_gs_ply_flag=bool(export_gs_ply_flag),
            export_gs_video_flag=bool(export_gs_video_flag),
        )
        downloads = [p for p in (res["glb_path"], res["ply_path"], res["npz_path"], res["poses_json_path"]) if p and os.path.isfile(p)]
        return (
            res["glb_path"],
            res["depth_vis_files"],
            res["gs_video_path"],
            downloads,
            res["poses_summary"],
        )
    except Exception as e:
        traceback.print_exc()
        return None, [], None, [], {"status": "error", "message": str(e)}

def _ui_run_video_or_folder(
    video_in,
    folder_path,
    model_id,
    video_fps,
    max_frames,
    process_res,
    ref_view_strategy,
    use_ray_pose,
    conf_thresh_percentile,
    num_max_points,
    show_cameras,
    export_gs_ply_flag,
    export_gs_video_flag,
    progress=gr.Progress(track_tqdm=True),
):
    src = video_in if video_in else (folder_path.strip() if folder_path else "")
    if not src:
        raise gr.Error("Please upload a video file or specify a folder path on disk.")
    try:
        progress(0.1, desc=f"Extracting views & running {model_id}...")
        res = run_da3_inference(
            input_source=src,
            model_id=model_id,
            process_res=int(process_res),
            ref_view_strategy=ref_view_strategy,
            use_ray_pose=bool(use_ray_pose),
            conf_thresh_percentile=float(conf_thresh_percentile),
            num_max_points=int(num_max_points),
            show_cameras=bool(show_cameras),
            export_gs_ply_flag=bool(export_gs_ply_flag),
            export_gs_video_flag=bool(export_gs_video_flag),
            video_fps=float(video_fps),
            max_frames=int(max_frames),
        )
        downloads = [p for p in (res["glb_path"], res["ply_path"], res["npz_path"], res["poses_json_path"]) if p and os.path.isfile(p)]
        return (
            res["glb_path"],
            res["depth_vis_files"],
            res["gs_video_path"],
            downloads,
            res["poses_summary"],
        )
    except Exception as e:
        traceback.print_exc()
        return None, [], None, [], {"status": "error", "message": str(e)}

with gr.Blocks(title="Depth Anything 3 — Any-View Depth, Pose, GLB & 3DGS", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🌊 ByteDance Depth Anything 3 (DA3) — Any-View Depth, Pose, `.glb` & 3DGS `.ply`\n"
        "Select between **`DA3NESTED-GIANT-LARGE-1.1`** (Flagship Metric + Pose + 3DGS), "
        "**`DA3METRIC-LARGE`** (Apache 2.0 Commercial-OK Metric Depth), and **`DA3-LARGE-1.1` / `DA3-BASE`**."
    )

    with gr.Row():
        model_selector = gr.Dropdown(
            choices=MODEL_CHOICES,
            value=DEFAULT_MODEL,
            label="Depth Anything 3 Model Checkpoint",
            info="DA3NESTED-GIANT-LARGE-1.1 combines Any-View Giant (pose + 3DGS) with Metric Large. DA3METRIC-LARGE and DA3-BASE are Apache 2.0 commercial-OK.",
        )

    with gr.Tabs():
        with gr.TabItem("1. Single Image / Multi-View Images"):
            with gr.Row():
                with gr.Column(scale=5):
                    single_img = gr.Image(
                        label="Single Input Image (Monocular Depth, 3D Point Cloud & 3DGS)",
                        type="filepath",
                        height=240,
                        info="Upload a single RGB image for monocular metric depth, 3D point cloud .glb, and 3DGS .ply.",
                    )
                    multi_files = gr.File(
                        file_count="multiple",
                        file_types=["image"],
                        label="Or Upload 2+ Multi-View Images (Overrides Single Image Above)",
                        info="Upload multiple unposed photos of a scene or object for multi-view consistent depth, camera pose estimation, and fused 3DGS/GLB.",
                    )
                    with gr.Accordion("Inference & Export Settings", open=True):
                        with gr.Row():
                            process_res = gr.Slider(
                                252, 1008, value=504, step=14,
                                label="Processing Resolution (px)",
                                info="Multiple of 14 (DINOv2 patch size). 504 is the upstream default.",
                            )
                            ref_view_strategy = gr.Dropdown(
                                choices=["saddle_balanced", "saddle_sim_range", "first", "middle"],
                                value="saddle_balanced",
                                label="Reference View Strategy",
                                info="How DA3 selects the canonical coordinate frame across multi-view inputs.",
                            )
                        with gr.Row():
                            use_ray_pose = gr.Checkbox(
                                value=False,
                                label="Use Ray-Based Pose Estimation",
                                info="Estimate camera poses from dense depth-ray predictions instead of the camera decoder head.",
                            )
                            show_cameras = gr.Checkbox(
                                value=True,
                                label="Draw Camera Wireframes in .glb",
                                info="Include color-coded camera frustum pyramids in the exported scene.glb.",
                            )
                        with gr.Row():
                            conf_thresh = gr.Slider(
                                0.0, 80.0, value=40.0, step=5.0,
                                label="Confidence Filter Percentile (%)",
                                info="Drop the lowest N% confidence pixels when constructing the .glb point cloud.",
                            )
                            num_max_points = gr.Slider(
                                100000, 2000000, value=1000000, step=100000,
                                label="Max 3D Points in .glb",
                                info="Downsample cap for the exported GLB point cloud.",
                            )
                        with gr.Row():
                            export_gs_ply_flag = gr.Checkbox(
                                value=True,
                                label="Export 3D Gaussian Splatting (.ply)",
                                info="Export 0000.ply (native 3DGS head on Giant/Nested models; unprojected 3DGS on Metric/Large/Base).",
                            )
                            export_gs_video_flag = gr.Checkbox(
                                value=False,
                                label="Render 3DGS Novel-View Video (.mp4)",
                                info="Render a camera trajectory video from predicted 3D Gaussians (requires Giant/Nested model + gsplat).",
                            )
                    btn_run_img = gr.Button("🚀 Reconstruct Depth, Poses, .glb & 3DGS .ply", variant="primary", size="lg")

                with gr.Column(scale=6):
                    out_glb = gr.Model3D(label="3D Point Cloud + Camera Frustums (scene.glb)", height=380)
                    out_gallery = gr.Gallery(label="Input vs. Colormap Depth Maps", columns=2, height=240)
                    out_video = gr.Video(label="3DGS Novel-View Trajectory Video (.mp4)", height=220)
                    out_files = gr.File(
                        label="Download Exports (scene.glb, 3DGS 0000.ply, results.npz, camera_poses.json)",
                        info="Click any file to download the 3D scene, Gaussian splats, or camera matrices.",
                    )
                    out_json = gr.JSON(label="Estimated Camera Poses & Depth Summary")

            if EXAMPLE_SCENES:
                gr.Examples(
                    examples=EXAMPLE_SCENES,
                    inputs=[single_img, model_selector],
                    label="Built-in Depth Anything 3 Example",
                )

            btn_run_img.click(
                fn=_ui_run_images,
                inputs=[
                    single_img, multi_files, model_selector, process_res,
                    ref_view_strategy, use_ray_pose, conf_thresh, num_max_points,
                    show_cameras, export_gs_ply_flag, export_gs_video_flag,
                ],
                outputs=[out_glb, out_gallery, out_video, out_files, out_json],
            )

        with gr.TabItem("2. Video / Image Folder Mode"):
            with gr.Row():
                with gr.Column(scale=5):
                    video_in = gr.Video(
                        label="Upload Input Video (.mp4, .mov)",
                        height=240,
                        info="Upload a video clip; DA3 extracts keyframes at the specified FPS and reconstructs a unified 3D scene.",
                    )
                    folder_path = gr.Textbox(
                        value="/content/Depth-Anything-3/assets/examples/SOH",
                        label="Or Image Folder Path on Colab Disk",
                        info="Directory of multi-view images (used when no video is uploaded above).",
                    )
                    with gr.Row():
                        video_fps = gr.Slider(
                            0.5, 15.0, value=2.0, step=0.5,
                            label="Video Frame Extraction FPS",
                            info="Frames per second sampled from the input video.",
                        )
                        max_frames = gr.Slider(
                            2, 64, value=16, step=2,
                            label="Max Frames / Views",
                            info="Maximum number of frames fed into the any-view transformer.",
                        )
                    with gr.Row():
                        process_res_v = gr.Slider(252, 1008, value=504, step=14, label="Processing Resolution (px)", info="Multiple of 14.")
                        ref_view_v = gr.Dropdown(["saddle_balanced", "saddle_sim_range", "first", "middle"], value="saddle_balanced", label="Reference View Strategy", info="Reference camera selection strategy.")
                    with gr.Row():
                        use_ray_v = gr.Checkbox(value=False, label="Use Ray-Based Pose", info="Use ray pose estimation.")
                        show_cam_v = gr.Checkbox(value=True, label="Show Cameras in .glb", info="Render camera frustum wireframes.")
                        export_ply_v = gr.Checkbox(value=True, label="Export 3DGS .ply", info="Export 3D Gaussian Splatting .ply.")
                        export_vid_v = gr.Checkbox(value=False, label="Render 3DGS Video", info="Render 3DGS novel-view video.")
                    conf_v = gr.Slider(0.0, 80.0, value=40.0, step=5.0, label="Confidence Percentile (%)", info="Confidence filter percentile.")
                    max_pts_v = gr.Slider(100000, 2000000, value=1000000, step=100000, label="Max 3D Points", info="Max points in scene.glb.")
                    btn_run_vid = gr.Button("🎬 Reconstruct 3D Scene from Video / Folder", variant="primary", size="lg")

                with gr.Column(scale=6):
                    out_glb_v = gr.Model3D(label="Fused Multi-View Point Cloud + Camera Poses (.glb)", height=380)
                    out_gallery_v = gr.Gallery(label="Per-Frame Colormap Depth Maps", columns=2, height=240)
                    out_video_v = gr.Video(label="3DGS Novel-View Video (.mp4)", height=220)
                    out_files_v = gr.File(label="Download Exports (.glb, .ply, .npz, .json)", info="Download all generated 3D and pose files.")
                    out_json_v = gr.JSON(label="Multi-View Camera Poses & Scene Metadata")

            btn_run_vid.click(
                fn=_ui_run_video_or_folder,
                inputs=[
                    video_in, folder_path, model_selector, video_fps, max_frames,
                    process_res_v, ref_view_v, use_ray_v, conf_v, max_pts_v,
                    show_cam_v, export_ply_v, export_vid_v,
                ],
                outputs=[out_glb_v, out_gallery_v, out_video_v, out_files_v, out_json_v],
            )

    demo.load(
        lambda: gr.Info("Depth Anything 3 Ready! Choose a checkpoint (DA3NESTED-GIANT-LARGE-1.1 or DA3METRIC-LARGE) and click Reconstruct."),
        inputs=None,
        outputs=None,
    )

clear_output()
demo.queue(max_size=4, concurrency_limit=1).launch(share=True, debug=False, show_error=True)

In [ ]:
# @title STEP 5 — Keep-Alive (Prevent Colab Disconnect)
# @markdown Pings the Colab kernel every 30 minutes to keep your interactive session alive.

import IPython
from IPython.display import display

KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("DA3 keep-alive:", new Date().toISOString());
        if (window.google && google.colab && google.colab.kernel) {
            google.colab.kernel.proxyPort(0).then(() => {}).catch(() => {});
        }
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print("[KeepAlive] Active — pinging Colab runtime every 30 minutes.")

In [ ]:
# @title STEP 6 — Quick Test (Multi-View / Single-Image to `.glb`, 3DGS `.ply` & Depth)
# @markdown Runs Depth Anything 3 directly in the notebook and displays download links (`FileLink`) for the `.glb` point cloud + camera frustums, 3DGS `.ply`, and `.npz`/`.json` camera poses.

import os, glob
from IPython.display import FileLink, display, Image as IPImage

INPUT_PATH = "/content/Depth-Anything-3/assets/examples/SOH"  # @param {type:"string"}
MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"  # @param ["depth-anything/DA3NESTED-GIANT-LARGE-1.1", "depth-anything/DA3METRIC-LARGE", "depth-anything/DA3-LARGE-1.1", "depth-anything/DA3-BASE", "depth-anything/DA3-GIANT-1.1"]
PROCESS_RES = 504  # @param {type:"slider", min:252, max:1008, step:14}
MAX_FRAMES = 8  # @param {type:"slider", min:1, max:32, step:1}
EXPORT_GS_PLY = True  # @param {type:"boolean"}

if not os.path.exists(INPUT_PATH):
    fallback_imgs = sorted(glob.glob("/content/Depth-Anything-3/assets/examples/**/*.*", recursive=True))
    INPUT_PATH = fallback_imgs[0] if fallback_imgs else ""

if not INPUT_PATH:
    raise SystemExit("[Step 6] No input found. Upload an image/video or check /content/Depth-Anything-3/assets/examples.")

print(f"[Step 6] Running {MODEL_ID} on: {INPUT_PATH}")
res = run_da3_inference(
    input_source=INPUT_PATH,
    model_id=MODEL_ID,
    process_res=PROCESS_RES,
    max_frames=MAX_FRAMES,
    export_gs_ply_flag=EXPORT_GS_PLY,
)

if res["depth_vis_files"]:
    display(IPImage(filename=res["depth_vis_files"][0]))

print(f"\n[Step 6] Completed in {res['elapsed_s']}s | Views: {res['poses_summary']['num_views']} | Metric: {res['poses_summary']['is_metric']}")
for label, fpath in [
    ("3D Scene + Cameras (.glb)", res["glb_path"]),
    ("3D Gaussian Splats (.ply)", res["ply_path"]),
    ("Depth & Poses (.npz)", res["npz_path"]),
    ("Camera Poses (.json)", res["poses_json_path"]),
]:
    if fpath and os.path.isfile(fpath):
        print(f"  -> {label}: {fpath}")
        display(FileLink(fpath))

In [ ]:
# @title STEP 7 — Batch Process Images, Multi-View Scene Folders, or Videos
# @markdown Iterates over all items in `BATCH_INPUT_DIR` (individual images, subdirectories of multi-view images, or video files) with per-item error isolation and `tqdm` progress reporting.

import os, glob, pathlib
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_INPUT_DIR = "/content/da3_inputs"  # @param {type:"string"}
BATCH_OUTPUT_DIR = "/content/drive/MyDrive/AEI_3D_Out/DA3_Batch"  # @param {type:"string"}
MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"  # @param ["depth-anything/DA3NESTED-GIANT-LARGE-1.1", "depth-anything/DA3METRIC-LARGE", "depth-anything/DA3-LARGE-1.1", "depth-anything/DA3-BASE", "depth-anything/DA3-GIANT-1.1"]
PROCESS_RES = 504  # @param {type:"slider", min:252, max:1008, step:14}
VIDEO_FPS = 2.0  # @param {type:"number"}
MAX_FRAMES = 16  # @param {type:"integer"}

if not os.path.exists("/content/drive/MyDrive") and BATCH_OUTPUT_DIR.startswith("/content/drive"):
    BATCH_OUTPUT_DIR = "/content/da3_batch_output"

os.makedirs(BATCH_INPUT_DIR, exist_ok=True)
os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

in_root = pathlib.Path(BATCH_INPUT_DIR)
items = [p for p in sorted(in_root.iterdir()) if not p.name.startswith(".")]
if not items:
    ex_root = pathlib.Path("/content/Depth-Anything-3/assets/examples")
    if ex_root.exists():
        items = [p for p in sorted(ex_root.iterdir()) if not p.name.startswith(".")][:3]
        print(f"[Batch] {BATCH_INPUT_DIR} is empty; using {len(items)} built-in example item(s) from {ex_root}")

print(f"[Batch] Processing {len(items)} item(s) with {MODEL_ID} -> {BATCH_OUTPUT_DIR}")
results = []

for idx, item_path in enumerate(tqdm(items, desc="DA3 Batch"), 1):
    slug = f"{idx:03d}_{item_path.stem}"
    item_out_dir = os.path.join(BATCH_OUTPUT_DIR, slug)
    expected_glb = os.path.join(item_out_dir, "scene.glb")
    if os.path.isfile(expected_glb) and os.path.getsize(expected_glb) > 1024:
        print(f"  [{idx}/{len(items)}] SKIP (already exists): {expected_glb}")
        results.append((slug, "skipped", expected_glb))
        continue
    try:
        out = run_da3_inference(
            input_source=str(item_path),
            model_id=MODEL_ID,
            process_res=PROCESS_RES,
            video_fps=VIDEO_FPS,
            max_frames=MAX_FRAMES,
            export_gs_ply_flag=True,
            output_dir=item_out_dir,
        )
        results.append((slug, "ok", out["glb_path"]))
    except Exception as e:
        print(f"  [{idx}/{len(items)}] ERROR on {item_path.name}: {e}")
        results.append((slug, "error", str(e)))
    finally:
        cleanup_vram()

ok_n = sum(1 for _, s, _ in results if s == "ok")
sk_n = sum(1 for _, s, _ in results if s == "skipped")
er_n = sum(1 for _, s, _ in results if s == "error")
print(f"\n[Batch Summary] {ok_n} succeeded, {sk_n} skipped, {er_n} failed. Output root: {BATCH_OUTPUT_DIR}")
for slug, status, glb_p in results:
    if status == "ok" and glb_p and os.path.isfile(glb_p):
        display(FileLink(glb_p))